# Analysis
In this notebook Bebras tasks will be assigned CT tags. (1) <br>
Afterwards, we will start analysing model performances on these tasks (2) by
- Calculating frequency of correct answers (with and without retry)
- Calculate baseline performances for each model
- Compare models amongst each other
- Possibly check whether task perfromance is mediated by certain tags

# 1. Task properties
We will create a data frame which contains all the necessary information about the tasks, including the CT skills. 

In [3]:
#Packages etc
from pathlib import Path
import os
from collections import defaultdict
import pandas as pd
import json

output_path =  Path(r"C:\Users\schul\Documents\uni\Master_Kogni\Praktikum\Tasks\output")


In [3]:

task_names = ["BeckyBee", "FlowerGarden", "StoneFactory", "Strawberries","TheGift", #7+8, easy
              "ClassroomSeating", "ColourTheFrog!", "OhridPearls", "Tic-Tac-Toe","WheatStorage", #7+8, medium
              "ConnectionOfIslands", "FavouriteMovie", "Mysteria", "Ordering1","UndergroundTrainNetwork", #7+8, hard
              "Cipher8", "InLove", "MarysNeighbours", "NutsAndBolts","RugWeaving", #9+10, easy
              "BeaverAI", "HangarCarousel", "OverlappingVillages", "ThePrinter", #9+10, medium
              "CollectingStones", "FavouriteGem", "FourTiles", "Maze", #9+10, hard
              "ColourfulCandles", "ListenAndWalk", "Lists", "MovieNight","Words", #11+12, easy
              "BeaverDam", "JumpingGame", "TreasureBox", #11+12, medium
              "BeaverDatabase", "BeaverGames", "Ordering2", "SeashellsAndPebbles","Virus"] #11+12, hard

# Will create an empty data frame and adapt the information manually
task_len = len(task_names)

data_empty = {"Task" : task_names,
              "Age": [None] * task_len, 
              "Difficulty" : [None] * task_len,
              "Decomposition": [True] * task_len, 
              "Pattern Recognition": [True] * task_len, 
              "Abstraction": [True] * task_len,
              "Modelling & Simulation" : [True] * task_len,
              "Algorithms": [True] * task_len, 
              "Evaluation": [True] * task_len 
              }

df_empty = pd.DataFrame(data_empty)




In [ ]:
df_empty = df_empty.transpose()
print(df_empty.head())

                              0             1             2             3   \
Task                    BeckyBee  FlowerGarden  StoneFactory  Strawberries   
Age                         None          None          None          None   
Difficulty                  None          None          None          None   
Decomposition               True          True          True          True   
Pattern Recognition         True          True          True          True   
Abstraction                 True          True          True          True   
Modelling & Simulation      True          True          True          True   
Algorithms                  True          True          True          True   
Evaluation                  True          True          True          True   

                             4                 5               6   \
Task                    TheGift  ClassroomSeating  ColourTheFrog!   
Age                        None              None            None   
Difficulty  

In [23]:
df_path = Path(r"C:\Users\schul\Documents\uni\Master_Kogni\Praktikum\Tasks")
df_empty.to_csv(df_path / "df_empty.csv", encoding = "utf-8", header = False)

I have manually filled in the categories and will load them here. Each category is listed and if a task fulfills this category, it is marked as TRUE 

In [ ]:
df_path = Path(r"C:\Users\schul\Documents\uni\Master_Kogni\Praktikum\Tasks")
df_categories = pd.read_csv(df_path / "df_categories.csv", sep = ";")
print(df_categories.head())

                     Task BeckyBee FlowerGarden StoneFactory Strawberries  \
0                     Age      7+8          7+8          7+8          7+8   
1              Difficulty     easy         easy         easy         easy   
2           Decomposition     TRUE         TRUE         TRUE         TRUE   
3     Pattern Recognition    FALSE         TRUE         TRUE        FALSE   
4             Abstraction    FALSE        FALSE         TRUE         TRUE   
5  Modelling & Simulation     TRUE         TRUE         TRUE         TRUE   
6              Algorithms     TRUE         TRUE         TRUE         TRUE   
7              Evaluation     TRUE         TRUE         TRUE         TRUE   

  TheGift ClassroomSeating ColourTheFrog! OhridPearls Tic-Tac-Toe  ...  \
0     7+8              7+8            7+8         7+8         7+8  ...   
1    easy           medium         medium      medium      medium  ...   
2    TRUE             TRUE           TRUE        TRUE        TRUE  ...   
3   FALSE 

**Accuracy Dataframe** <br>
We also need to save a dataframe which contains all the information about each models accuracy, we just can also save this in the previous data frame for simplicity reasons

In [ ]:
# Default values for now are 0

full_df = df_categories

models = ["Gemini_2.5","Gemini_2.5_retry",
          "Gpt_4o", "Gpt_4o_retry",
          "Gpt_5", "Gpt_5_retry"]

new_rows = []
for model in models:
    row = {"Task": model}
    for col in task_names:
        row[col] = 0
    new_rows.append(row)

models_df = pd.DataFrame(new_rows)
full_df = pd.concat([full_df, models_df], ignore_index= True)

print(full_df.head())

Now iterate through the JSON files and adapt the points based<br>
on task difficulty:<br> <br>
Easy: 6pts<br>
Medium: 9pts<br>
Hard: 12pts<br>
... if the answers are correct

In [ ]:
# Mini helper to determine points
def difficultyPoints(difficulty):
    if difficulty == "easy":
        return 6
    elif difficulty == "medium":
        return 9
    elif difficulty == "hard":
        return 12
        
age_groups = ["year_7_8", "year_9_10", "year_11_12"]
difficulties = ["easy", "medium", "hard"]
#reminder: task_names, models

for age_group in age_groups:
    for difficulty in difficulties:
        for model in models:
            # Handle retry naming and file matching
            is_retry = model.endswith("_retry")
            base_model = model.replace("_retry", "")

            # Folder path corresponds to the non-retry model
            model_folder = os.path.join(output_path, age_group, difficulty, base_model)
            if not os.path.exists(model_folder):
                continue

            for task in task_names:
                # Choose correct filename pattern
                if is_retry:
                    json_file = os.path.join(model_folder, f"{task}_{base_model}_retry.json")
                else:
                    json_file = os.path.join(model_folder, f"{task}_{base_model}.json")

                # Skip if JSON doesn’t exist
                if not os.path.exists(json_file):
                    continue

                # Read JSON safely
                try:
                    with open(json_file, "r") as f:
                        data = json.load(f)
                except json.JSONDecodeError:
                    print(f"Invalid JSON: {json_file}")
                    continue

                # Check correctness and assign points
                if data.get("is_correct") == True:
                    points = difficultyPoints(difficulty)
                    full_df.loc[full_df["Task"] == model, task] = points
                else:
                    # remains 0
                    pass

print(full_df.head())

                      Task BeckyBee FlowerGarden StoneFactory Strawberries  \
0                      Age      7+8          7+8          7+8          7+8   
1               Difficulty     easy         easy         easy         easy   
2            Decomposition     TRUE         TRUE         TRUE         TRUE   
3      Pattern Recognition    FALSE         TRUE         TRUE        FALSE   
4              Abstraction    FALSE        FALSE         TRUE         TRUE   
5   Modelling & Simulation     TRUE         TRUE         TRUE         TRUE   
6               Algorithms     TRUE         TRUE         TRUE         TRUE   
7               Evaluation     TRUE         TRUE         TRUE         TRUE   
8               Gemini_2.5        0            6            6            6   
9         Gemini_2.5_retry        0            0            0            0   
10                  Gpt_4o        0            6            6            0   
11            Gpt_4o_retry        0            0            0   

In [ ]:
print(full_df.head())
full_df.to_csv(df_path / "full_df.csv", encoding = "utf-8", header= True)

                      Task BeckyBee FlowerGarden StoneFactory Strawberries  \
0                      Age      7+8          7+8          7+8          7+8   
1               Difficulty     easy         easy         easy         easy   
2            Decomposition     TRUE         TRUE         TRUE         TRUE   
3      Pattern Recognition    FALSE         TRUE         TRUE        FALSE   
4              Abstraction    FALSE        FALSE         TRUE         TRUE   
5   Modelling & Simulation     TRUE         TRUE         TRUE         TRUE   
6               Algorithms     TRUE         TRUE         TRUE         TRUE   
7               Evaluation     TRUE         TRUE         TRUE         TRUE   
8               Gemini_2.5        0            6            6            6   
9         Gemini_2.5_retry        0            0            0            0   
10                  Gpt_4o        0            6            6            0   
11            Gpt_4o_retry        0            0            0   

# 2. Performance analysis 

In this step we will determine the accuracy based on model/ difficulty/ age/ categories etc.

In [20]:
#so that we don't have to re-run every piece of code
df_path = Path(r"C:\Users\schul\Documents\uni\Master_Kogni\Praktikum\Tasks")
full_df = pd.read_csv(df_path / "full_df.csv", sep = ",")
full_df = full_df.iloc[: , 1:]
full_df = full_df.set_index("Task")
print(full_df.head())

                    BeckyBee FlowerGarden StoneFactory Strawberries TheGift  \
Task                                                                          
Age                      7+8          7+8          7+8          7+8     7+8   
Difficulty              easy         easy         easy         easy    easy   
Decomposition           TRUE         TRUE         TRUE         TRUE    TRUE   
Pattern Recognition    FALSE         TRUE         TRUE        FALSE   FALSE   
Abstraction            FALSE        FALSE         TRUE         TRUE    TRUE   

                    ClassroomSeating ColourTheFrog! OhridPearls Tic-Tac-Toe  \
Task                                                                          
Age                              7+8            7+8         7+8         7+8   
Difficulty                    medium         medium      medium      medium   
Decomposition                   TRUE           TRUE        TRUE        TRUE   
Pattern Recognition             TRUE          FALSE

In [21]:
print(full_df)

                       BeckyBee FlowerGarden StoneFactory Strawberries  \
Task                                                                     
Age                         7+8          7+8          7+8          7+8   
Difficulty                 easy         easy         easy         easy   
Decomposition              TRUE         TRUE         TRUE         TRUE   
Pattern Recognition       FALSE         TRUE         TRUE        FALSE   
Abstraction               FALSE        FALSE         TRUE         TRUE   
Modelling & Simulation     TRUE         TRUE         TRUE         TRUE   
Algorithms                 TRUE         TRUE         TRUE         TRUE   
Evaluation                 TRUE         TRUE         TRUE         TRUE   
Gemini_2.5                    0            6            6            6   
Gemini_2.5_retry              0            0            0            0   
Gpt_4o                        0            6            6            0   
Gpt_4o_retry                  0       

In [40]:
# Small helper function to calculate all points
# If 

def accuracyFor(model, age=None, difficulty=None):
    # Handle multiple ages or difficulties
    age_mask = full_df.loc["Age"].isin(age) if isinstance(age, list) else (full_df.loc["Age"] == age)
    diff_mask = full_df.loc["Difficulty"].isin(difficulty) if isinstance(difficulty, list) else (full_df.loc["Difficulty"] == difficulty)

    vals = full_df.loc[model, age_mask & diff_mask]
    points = pd.to_numeric(vals).sum()
    return points



In [41]:
# Function tests
print(accuracyFor("Gpt_4o", age ="9+10", difficulty="medium"))
print(accuracyFor("Gpt_4o", age="7+8"))
print(accuracyFor("Gpt_4o", difficulty="medium"))
print(accuracyFor("Gpt_4o", age =["7+8","9+10"], difficulty="medium"))

9
0
0
18


18
